In [1]:
#RNN


from tensorflow import keras
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding,SimpleRNN,Bidirectional,LSTM,Dropout
from tensorflow.keras.datasets import imdb
from tensorflow.keras import initializers
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.preprocessing.text import text_to_word_sequence



In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
max_features = 20000

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

In [4]:
maxlen = 200

In [5]:
max_len = 200
x_train = pad_sequences(x_train,maxlen=max_len)
x_test = pad_sequences(x_test,maxlen=max_len)

In [6]:
rnn_hidden_dim = 5
word_embedding_dim = 50

model_rnn = Sequential()

model_rnn.add(
    Embedding(
        max_features,
        word_embedding_dim
    )
)

model_rnn.add(
    SimpleRNN(
        rnn_hidden_dim,
        kernel_initializer= initializers.RandomNormal(stddev=0.001),
        recurrent_initializer=initializers.Identity(gain=1.0),
        activation='relu',
        input_shape=x_train.shape[1:]
    )
)

model_rnn.add(Dense(1,activation='sigmoid'))

In [7]:
model_rnn.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 50)          1000000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 5)                 280       
                                                                 
 dense (Dense)               (None, 1)                 6         
                                                                 
Total params: 1000286 (3.82 MB)
Trainable params: 1000286 (3.82 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [8]:
rmpprop = keras.optimizers.RMSprop(learning_rate=0.0001)

model_rnn.compile(
    loss='binary_crossentropy',
    optimizer=rmpprop,
    metrics=['accuracy']
)

In [ ]:
model_rnn.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=10,
    validation_split=0.2,
    validation_data=(x_test, y_test)
)

Epoch 1/10


782/782 [==============================] - 32s 38ms/step - loss: 0.6414 - accuracy: 0.6166 - val_loss: 0.5776 - val_accuracy: 0.7003
Epoch 2/10
782/782 [==============================] - 29s 37ms/step - loss: 0.4397 - accuracy: 0.8084 - val_loss: 0.3971 - val_accuracy: 0.8170
Epoch 3/10
782/782 [==============================] - 29s 37ms/step - loss: 0.3340 - accuracy: 0.8634 - val_loss: 0.3876 - val_accuracy: 0.8319
Epoch 4/10
782/782 [==============================] - 29s 38ms/step - loss: 0.2861 - accuracy: 0.8880 - val_loss: 0.3390 - val_accuracy: 0.8584
Epoch 5/10
782/782 [==============================] - 27s 35ms/step - loss: 0.2555 - accuracy: 0.9028 - val_loss: 0.3304 - val_accuracy: 0.8599
Epoch 6/10
782/782 [==============================] - 27s 35ms/step - loss: 0.2334 - accuracy: 0.9109 - val_loss: 0.3374 - val_accuracy: 0.8572
Epoch 7/10
782/782 [==============================] - 27s 35ms/step - loss: 0.2177 - accuracy: 0.9192 - val_loss: 0.3395 - val_accurac

In [10]:
score,acc = model_rnn.evaluate(
    x_test,
    y_test,
    batch_size=32
)

print(
    f'Test Score: {score}\n'
    f'Accuracy:{acc}'
)

782/782 [==============================] - 6s 7ms/step - loss: 0.3638 - accuracy: 0.8447
Test Score: 0.36375707387924194
Accuracy:0.8447200059890747


In [11]:
word_index = imdb.get_word_index()
index_offset =3

def encode_review(text):
    tokens= text_to_word_sequence(text.lower())
    seq= [
        word_index.get(t,2) + index_offset
        for t in tokens
        if (word_index.get(t,2)+index_offset) < max_features
    ]
    return seq

In [ ]:
def predict_reviews(reviews, threshold=0.5):
    seqs = [encode_review(r) for r in reviews]
    padded = pad_sequences(seqs, maxlen=max_len)
    probs = model_rnn.predict(padded).ravel()   # ← model_rnn
    labels = (probs >= threshold).astype(int)
    
    for r, p, l in zip(reviews, probs, labels):
        print(f'{p:0.3f} ({"pos" if l else "neg"}) -> {r}')

In [13]:
movie_reviews = [
    "Great acting, boring storyline",
    "Awesome cast, weak direction, bad movie",
    "Nice music, poor screenplay, bad plot",
    "Good plot, bad execution sadly",
    "Loved visuals, hated the pace, narrow plot"
]

predict_reviews(movie_reviews)

NameError: name 'model' is not defined